# Configuración ambiente de trabajo

## Librerias

In [1]:
!pip install --upgrade "numpy<2.0.0"
!pip install surprise
!pip install fastparquet


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np

from pathlib import Path
import joblib
import json
from tqdm import tqdm
import time


from surprise import Dataset as SurpriseDataset
from surprise import Reader
from surprise import SVDpp
from surprise import accuracy

from sklearn.preprocessing import MinMaxScaler

from typing import Optional, Union

## Colab
Configuración para correr local o en google colab.

In [3]:
from pathlib import Path
import sys
import os

is_colab =  "/content" in str(Path.cwd())
if is_colab:
  print(f"Running in Google Colab")
  from google.colab import drive
  drive.mount('/content/drive')
  new_pwd = Path("/content/drive/MyDrive/CLASES MAESTRIA/MINE4201-SISTEMAS-DE-RECOMENDACION/TALLER2/A-Recommender-System-for-Yelp-Reviews/notebooks")
  %cd {new_pwd}

  # Explicitly set the current working directory to the notebook's directory
  os.chdir(new_pwd)

  if Path.cwd() != new_pwd:
    raise Exception("Could not change working directory to %s" % new_pwd)

else:
  print(f"Running locally")
  # Para evitar recargar el kernel. Cada vez que ejecuto una celda, se vuelve a cargar utils.py
  %load_ext autoreload
  %autoreload 2

Running locally


# Cargar los datos

In [4]:
# ── Paths
ROOT       = Path('..')
DATA_DIR   = ROOT / 'data'
RAW_DIR   = DATA_DIR / 'raw' # yelp JSONs live here
PROCESSED_DIR = DATA_DIR / 'processed' # processed data will be saved here
MODEL_DIR = ROOT / 'models' # trained model will be saved here

# create directories if they don't exist
for directory in [DATA_DIR, RAW_DIR, PROCESSED_DIR, MODEL_DIR]:
    if not directory.exists():
        directory.mkdir()

In [5]:
# "Quiero comer"
FOOD_INTENT = {
    "restaurants",
    "food",
    "pizza",
    "burgers",
    "mexican",
    "italian",
    "sushi_bars",
    "japanese",
    "chinese",
    "thai",
    "korean",
    "vietnamese",
    "seafood",
    "steakhouses",
    "barbeque",
    "sandwiches",
    "salad",
    "tacos",
    "ramen",
    "poke",
    "diners",
    "delis",
    "food_trucks",
    "fast_food",
    "breakfast_&_brunch",
    "specialty_food",
    "chicken_wings",
    "food_delivery_services",
    "middle_eastern",
    "caribbean",
    'local_flavor',
}

# "Quiero un café o algo ligero"
CAFE_SNACK_INTENT = {
    "coffee_&_tea",
    "cafes",
    "bakeries",
    "bubble_tea",
    "desserts",
    "ice_cream_&_frozen_yogurt",
    "donuts",
    "bagels",
    "juice_bars_&_smoothies",
    "tea_rooms",
    "gelato",
    "macarons",
    "sandwiches"
}

# Quiero algo familiar
FAMILY_INTENT = {
    "kids_activities",
    "playgrounds",
    "children's_museums",
    "parks",
    "trampoline_parks",
    "museums",
    "zoos",
    "petting_zoos",
    "amusement_parks",
    "mini_golf",
    "bowling",
    "ice_cream_&_frozen_yogurt",
    "pizza",
    "books",
    "pets"
}


# Quiero hacer ejercicio o alguna actividad
ACTIVE_LIFE_INTENT = {
    "active_life",
    "gyms",
    "yoga",
    "pilates",
    "parks",
    "hiking",
    "bike_rentals",
    "bikes",
    "cycling_classes",
    "martial_arts",
    "boxing",
    "climbing",
    "rock_climbing",
    "swimming_pools",
    "golf",
    "tennis",
    "pickleball",
    "shopping",
    "beauty_&_spas",
    "arts_&_entertainment",
    "massage",
    "books",
    "thrift_stores",
}

NIGHTLIFE_INTENT = {
    "nightlife",
    "bars",
    "beer",
    "lounges",
    "karaoke",
    "comedy_clubs",
    "sports_bars",
    "pubs",
    "speakeasies",
    "wine_bars",
    "wine_&_spirits",
    "cocktail_bars",
    "music_venues"
}

SERVICES_INTENT = {
    'home_services', 'automotive', 'health_&_medical', 'local_services',
       'auto_repair', 'hotels_&_travel', 'event_planning_&_services',
       'real_estate', 'doctors', 'hotels', 'dentists', 'tires',
       'oil_change_stations', 'professional_services', 'general_dentistry',
       'apartments', 'auto_parts_&_supplies', 'contractors', 'car_dealers',
       'cosmetic_dentists', 'financial_services', 'public_services_&_government',
       'banks_&_credit_unions', 'education', 'insurance',
       'religious_organizations', 'libraries', 'churches', 'auto_insurance',
       'specialty_schools', 'life_insurance',
       'home_&_rental_insurance', 'post_offices', 'colleges_&_universities',
       'landmarks_&_historical_buildings', 'mass_media',
       'departments_of_motor_vehicles', 'elementary_schools', "beauty_&_spas"}

PREFERRED_CATEGORIES = FOOD_INTENT | CAFE_SNACK_INTENT | FAMILY_INTENT | ACTIVE_LIFE_INTENT | NIGHTLIFE_INTENT | SERVICES_INTENT

# Modelo basado en contexto

In [6]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV
import numpy as np
import pandas as pd
import joblib

from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import MultiLabelBinarizer

# modelo de ejemplo con SVD++
model_svd = joblib.load(MODEL_DIR / 'model_SVDpp_100.joblib')

#df_businesses = pd.read_parquet(RAW_DIR / 'yelp_academic_dataset_business.parquet', engine='fastparquet')
df_businesses = pd.read_parquet(PROCESSED_DIR / 'business.parquet', engine='fastparquet')
print(f"hay {df_businesses.shape[0]} filas en businesses")
print(df_businesses.columns, "\n")
#df_reviews = pd.read_parquet(RAW_DIR / 'yelp_academic_dataset_review.parquet', engine='fastparquet')
df_reviews = pd.read_parquet(PROCESSED_DIR / 'reviews.parquet', engine='fastparquet')
print(f"hay {df_reviews.shape[0]} filas en reviews")
print(df_reviews.columns, "\n")



INTENT_GROUPS = {
    "food": FOOD_INTENT | CAFE_SNACK_INTENT,
    "family": FAMILY_INTENT,
    "active_life": ACTIVE_LIFE_INTENT,
    "nightlife": NIGHTLIFE_INTENT,
    "services": SERVICES_INTENT,
}


def clean_categories(x):
    if not isinstance(x, str):
        return []
    
    return [
        category.strip().lower().replace(" ", "_")
        for category in x.split(",")
        if category.strip() != ""
    ]


def has_any_category(categories, category_set):
    return int(any(category in category_set for category in categories))


# Unir con contexto
df_reviews = df_reviews.merge(
    df_businesses[
        [
            "business_id",
            "city",
            "categories",
            "latitude",
            "longitude",
            "review_count"
        ]
    ],
    on="business_id",
    how="left"
)

# Orden temporal
df_reviews = df_reviews.sort_values("date").reset_index(drop=True)


# Crear 5 variables de intención
df_reviews["categories_list"] = df_reviews["categories"].fillna("").apply(clean_categories)

for intent_name, intent_categories in INTENT_GROUPS.items():
    df_reviews[f"intent_{intent_name}"] = df_reviews["categories_list"].apply(
        lambda categories: has_any_category(categories, intent_categories)
    )

df_reviews = df_reviews.drop(columns=["categories", "categories_list"])


# Predicción base SVD++
def calculate_cf_score(row, model_svd):
    return model_svd.predict(row["user_id"], row["business_id"]).est


print("Generando pred_rating con SVD++...")
df_reviews["pred_rating"] = df_reviews.apply(
    lambda row: calculate_cf_score(row, model_svd),
    axis=1
)

# Selección de variables
target_col = "stars"

num_features = [
    "pred_rating",
    "longitude",
    "latitude",
    "review_count"
] 


intent_features = [
    col for col in df_reviews.columns
    if col.startswith("intent_")
]

columns = [target_col] + num_features + intent_features
df_final = df_reviews[columns].copy()

# Split temporal
n = len(df_final)

df_train = df_final.iloc[:int(n * 0.80)].copy()
df_test = df_final.iloc[int(n * 0.80):].copy()

X_train = df_train.drop(columns=[target_col])
y_train = df_train[target_col]

X_test = df_test.drop(columns=[target_col])
y_test = df_test[target_col]


# Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("intent", "passthrough", intent_features)
    ]
)

pipe_context = Pipeline(steps=[
    ("prep", preprocessor),
    ("poly", PolynomialFeatures(degree=1, include_bias=False)),
    ("regressor", LinearRegression())
])


# Grid con LinearRegression, Ridge, Lasso y ElasticNet
param_grid = [
    {
        "prep__num": [StandardScaler(), MinMaxScaler()],
        "poly__degree": [1],
        "regressor": [LinearRegression()]
    },
    {
        "prep__num": [StandardScaler(), MinMaxScaler()],
        "poly__degree": [1],
        "regressor": [Ridge()],
        "regressor__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]
    },
    {
        "prep__num": [StandardScaler(), MinMaxScaler()],
        "poly__degree": [1],
        "regressor": [Lasso(max_iter=10000)],
        "regressor__alpha": [0.0001, 0.001, 0.01, 0.1, 1.0]
    },
    {
        "prep__num": [StandardScaler(), MinMaxScaler()],
        "poly__degree": [1],
        "regressor": [ElasticNet(max_iter=10000)],
        "regressor__alpha": [0.0001, 0.001, 0.01, 0.1, 1.0],
        "regressor__l1_ratio": [0.2, 0.5, 0.8]
    }
]


grid_search = GridSearchCV(
    pipe_context,
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    verbose=1,
    n_jobs=2
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

print("Mejor modelo:")
print(grid_search.best_params_)

hay 114658 filas en businesses
Index(['business_id', 'name', 'address', 'city', 'state', 'postal_code',
       'latitude', 'longitude', 'stars', 'review_count', 'is_open',
       'categories', 'attributes.ByAppointmentOnly',
       'attributes.BusinessAcceptsCreditCards', 'attributes.BikeParking',
       'attributes.RestaurantsPriceRange2', 'attributes.CoatCheck',
       'attributes.RestaurantsTakeOut', 'attributes.RestaurantsDelivery',
       'attributes.Caters', 'attributes.WiFi', 'attributes.BusinessParking',
       'attributes.WheelchairAccessible', 'attributes.HappyHour',
       'attributes.OutdoorSeating', 'attributes.HasTV',
       'attributes.RestaurantsReservations', 'attributes.DogsAllowed',
       'attributes.Alcohol', 'attributes.GoodForKids',
       'attributes.RestaurantsAttire', 'attributes.Ambience',
       'attributes.RestaurantsTableService',
       'attributes.RestaurantsGoodForGroups', 'attributes.DriveThru',
       'attributes.NoiseLevel', 'attributes.GoodForMeal',

In [7]:
# Baseline: solo SVD++
rmse_base = np.sqrt(mean_squared_error(y_test, X_test["pred_rating"]))
r2_base = r2_score(y_test, X_test["pred_rating"])

# Modelo contextual
y_pred_context = best_model.predict(X_test)

rmse_context = np.sqrt(mean_squared_error(y_test, y_pred_context))
r2_context = r2_score(y_test, y_pred_context)

print("BASE SVD++")
print("RMSE:", rmse_base)
print("R2:", r2_base)

print("MODELO CONTEXTUAL")
print("RMSE:", rmse_context)
print("R2:", r2_context)

print("Mejora RMSE:", rmse_base - rmse_context)
print("Mejora R2:", r2_context - r2_base)
# 6. PESOS DEL CONTEXTO (PESOS ENTRENADOS)
# ---------------------------------------------------------
weights = best_model.named_steps['regressor'].coef_
# Obtenemos nombres de las columnas transformadas
feature_names = best_model.named_steps['prep'].get_feature_names_out()
final_feature_names = best_model.named_steps['poly'].get_feature_names_out(feature_names)

pesos_df = pd.DataFrame({
    'Feature': final_feature_names, 
                         'Weight': weights})
pesos_df = pesos_df.sort_values(by='Weight', ascending=False)

print("\n--- PESOS DE LAS VARIABLES EN LA PREDICCIÓN FINAL ---")
display(pesos_df.head(15))
# guardar modelo

joblib.dump(best_model, MODEL_DIR / 'model_contextual.joblib')

BASE SVD++
RMSE: 1.045650187861196
R2: 0.5724865404501904
MODELO CONTEXTUAL
RMSE: 0.9954182238366368
R2: 0.6125745666124112
Mejora RMSE: 0.050231964024559095
Mejora R2: 0.040088026162220736

--- PESOS DE LAS VARIABLES EN LA PREDICCIÓN FINAL ---


,Feature,Weight
0,num__pred_rating,1.097969
8,intent__intent_services,0.024008
1,num__longitude,0.004303
2,num__latitude,0.002648
7,intent__intent_nightlife,-0.001053
5,intent__intent_family,-0.012929
3,num__review_count,-0.029778
6,intent__intent_active_life,-0.041049
4,intent__intent_food,-0.050832


['../models/model_contextual.joblib']

# Business Context Recomendation

La estrategia planteada es:
1. Filtrar negocios no vistos por el usuario y que esten abiertos.
2. Si aplica, filtrar ciudades preferidas.
3. Si aplica, filtrar categorias preferidas.
4. Si aplica, filtrar dia/hora preferido.
5. Calcular predicción del modelo SVD++.
6. Calcular score de popularidad.
7. Calcure score por ciudad.
8. Calcular time-context score.
9. Calcular categories-score.
10. Ponderar score final.
11. Reordenar ranking de recomendación.

### Carga de datos

Para esta parte ya no se va a usar un sample del 10%, si no que se van a cargar todos los datos.


Dado que los archivos son gigantes, se decidió cargar por chunks (1 millón de lineas cada uno, en total son alrededor de 7 chunks para el archivo más grande), y luego guardar en formato parquet para luego re-cargarlo completo en parquet, parece redundante pero ahorra mucha ram y tiempo en los archivos pesados


In [8]:
raw_json_paths = list(RAW_DIR.glob('*.json'))
create_parquets = any(not (RAW_DIR / f"{json_path.stem}.parquet").exists() for json_path in raw_json_paths)
if create_parquets:
    # --- IGNORE ---
    for json_path in raw_json_paths:
        with open(json_path, 'r') as f:

            chunksize = 1e6
            yelp_data = []
            yelp_chunks = pd.read_json(json_path, lines=True, chunksize=chunksize)
            for chunk in tqdm(yelp_chunks):
                yelp_data.append(chunk)

            # guardar en parquet
            pd.concat(yelp_data).to_parquet(Path(json_path.parent,f"{json_path.stem}.parquet"), index=False)
            #yelp_data:pd.DataFrame = pd.read_parquet(Path(json_path.parent,f"{json_path.stem}.parquet"))
else:
    print("Parquet files already exist. Skipping JSON to Parquet conversion.")
    print("Loading businesses")
    df_businesses = pd.read_parquet(RAW_DIR / 'yelp_academic_dataset_business.parquet', engine='fastparquet')
    print(f"hay {df_businesses.shape[0]} filas en businesses")
    print(df_businesses.columns, "\n")

    print("Loading reviews")
    df_reviews = pd.read_parquet(RAW_DIR / 'yelp_academic_dataset_review.parquet', engine='fastparquet')
    print(f"hay {df_reviews.shape[0]} filas en reviews")
    print(df_reviews.columns, "\n")

    print("Loading users")
    df_users = pd.read_parquet(RAW_DIR / 'yelp_academic_dataset_user.parquet', engine='fastparquet')
    print(f"hay {df_users.shape[0]} filas en users")
    print(df_users.columns, "\n")
hours_columns = list(df_businesses.filter(regex='hours.').columns)

# Diccionario con los horarios de cada negocio
df_businesses["hours"] = df_businesses.apply(lambda row: {day.split(".")[1]: row[day] for day in hours_columns if pd.notna(row[day])}, axis=1)
df_businesses[["name", "hours"] + hours_columns ].head()
attributes_columns = df_businesses.filter(regex='attributes.').columns
print(attributes_columns)
df_businesses[["name"] + list(attributes_columns)].head()
df_reviews["stars"].value_counts()

Parquet files already exist. Skipping JSON to Parquet conversion.
Loading businesses
hay 150346 filas en businesses
Index(['business_id', 'name', 'address', 'city', 'state', 'postal_code',
       'latitude', 'longitude', 'stars', 'review_count', 'is_open',
       'categories', 'attributes.ByAppointmentOnly',
       'attributes.BusinessAcceptsCreditCards', 'attributes.BikeParking',
       'attributes.RestaurantsPriceRange2', 'attributes.CoatCheck',
       'attributes.RestaurantsTakeOut', 'attributes.RestaurantsDelivery',
       'attributes.Caters', 'attributes.WiFi', 'attributes.BusinessParking',
       'attributes.WheelchairAccessible', 'attributes.HappyHour',
       'attributes.OutdoorSeating', 'attributes.HasTV',
       'attributes.RestaurantsReservations', 'attributes.DogsAllowed',
       'attributes.Alcohol', 'attributes.GoodForKids',
       'attributes.RestaurantsAttire', 'attributes.Ambience',
       'attributes.RestaurantsTableService',
       'attributes.RestaurantsGoodForGroup

stars
5    3231627
4    1452918
1    1069561
3     691934
2     544240
Name: count, dtype: int64

### Filtro Horario

Funciones auxiliares para evaluar si un negocio esta abierto a cierto horario (dia y hora)

In [9]:
from typing import Optional

def _is_hour_inside_range(time_range: str, hour: int) -> bool:
    """
    Evalúa si una hora entera está dentro de un rango tipo '10:00-21:00'.
    También maneja rangos que cruzan medianoche, como '18:00-02:00'.
    """

    if not isinstance(time_range, str) or "-" not in time_range:
        return True

    start_time, end_time = time_range.split("-")

    start_hour = int(start_time.split(":")[0])
    end_hour = int(end_time.split(":")[0])

    # Abierto 24 horas, ejemplo: 0:00-0:00
    if start_hour == end_hour:
        return True

    # Caso normal: 10:00-21:00
    if start_hour < end_hour:
        return start_hour <= hour < end_hour

    # Caso cruza medianoche: 18:00-02:00
    return hour >= start_hour or hour < end_hour


def is_open_at_context(
    hours_dict,
    day_list: Optional[list[str]] = None,
    hour: Optional[int] = None,
    filter_by_day: bool = False,
    filter_by_hour: bool = False
) -> bool:
    """
    Permite filtrar por día, por hora, por ambos o por ninguno.
    """

    # Si no quiero filtrar por día ni por hora, no excluyo el negocio
    if not filter_by_day and not filter_by_hour:
        return True

    # Si no hay diccionario de horarios, no excluyo el negocio
    if not isinstance(hours_dict, dict):
        return True

    # Caso 1: filtrar por día, pero no por hora
    if filter_by_day and not filter_by_hour:
        if day_list is None:
            return True
        # Si el día es una lista, revisamos si alguno de los días está en el diccionario de horarios
        return any(d in hours_dict for d in day_list)

    # Caso 2: filtrar por hora, pero no por día
    # Aquí revisamos si está abierto a esa hora en al menos un día
    if filter_by_hour and not filter_by_day:
        if hour is None:
            return True

        return any(
            _is_hour_inside_range(time_range, hour)
            for time_range in hours_dict.values()
        )

    # Caso 3: filtrar por día y por hora
    if filter_by_day and filter_by_hour:
        if day_list is None or hour is None:
            return True
        open_at_day = any(d in hours_dict for d in day_list)
        open_at_day_and_hour = False
        if open_at_day:
            open_at_day_and_hour = any(_is_hour_inside_range(hours_dict[day], hour) for day in day_list)
        return open_at_day_and_hour

    return True

### Boost por Ciudad

In [10]:

def city_match_score(
    city: str,
    preferred_cities = None
) -> float:
    
    if preferred_cities is None:
        return 1.0
    
    if isinstance(preferred_cities, str):
        preferred_cities = [preferred_cities]
    
    preferred_cities = [c.lower().strip() for c in preferred_cities]
    
    if str(city).lower().strip() in preferred_cities:
        return 1.1  # boost de 10% si la ciudad coincide
    
    return 0.0

### Boost contextual por tiempo

In [11]:
TIME_CONTEXT = {
    "morning": {
        "hours": range(6, 11),
        "boost_categories": {
            "breakfast_&_brunch": 1.5,
            "coffee_&_tea": 1.5,
            "cafes": 1.4,
            "bakeries": 1.4,
            "bagels": 1.3,
            "donuts": 1.3,
            "juice_bars_&_smoothies": 1.2
        }
    },
    "lunch": {
        "hours": range(11, 15),
        "boost_categories": {
            "restaurants": 1.2,
            "sandwiches": 1.4,
            "salad": 1.3,
            "soup": 1.2,
            "fast_food": 1.2,
            "food_trucks": 1.3,
            "tacos": 1.3,
            "poke": 1.3,
            "sushi_bars": 1.2
        }
    },
    "afternoon": {
        "hours": range(15, 18),
        "boost_categories": {
            "coffee_&_tea": 1.4,
            "cafes": 1.3,
            "desserts": 1.4,
            "ice_cream_&_frozen_yogurt": 1.4,
            "bakeries": 1.2,
            "bubble_tea": 1.3,
            "juice_bars_&_smoothies": 1.2
        }
    },
    "dinner": {
        "hours": range(18, 23),
        "boost_categories": {
            "restaurants": 1.2,
            "italian": 1.3,
            "sushi_bars": 1.3,
            "japanese": 1.2,
            "mexican": 1.2,
            "thai": 1.2,
            "steakhouses": 1.4,
            "seafood": 1.3,
            "mediterranean": 1.2,
            "pizza": 1.2,
            "barbeque": 1.3,
            "french": 1.3
        }
    },
    "late_night": {
        "hours": list(range(23, 24)) + list(range(0, 6)),
        "boost_categories": {
            "pizza": 1.5,
            "fast_food": 1.4,
            "bars": 1.4,
            "pubs": 1.3,
            "sports_bars": 1.3,
            "lounges": 1.2,
            "food_trucks": 1.2,
            "diners": 1.3
        }
    }
}

def get_time_bucket(hour):
    if 6 <= hour < 11:
        return "morning"
    elif 11 <= hour < 15:
        return "lunch"
    elif 15 <= hour < 18:
        return "afternoon"
    elif 18 <= hour < 23:
        return "dinner"
    else:
        return "late_night"
    

def context_score_by_hour(cat_list, hour):
    bucket = get_time_bucket(hour)
    boost_dict = TIME_CONTEXT[bucket]["boost_categories"]

    boost = 1.0

    for cat in cat_list:
        if cat in boost_dict:
            #print(f"Category '{cat}' gets a boost of {boost_dict[cat]} in the '{bucket}' time bucket.")
            boost = max(boost, boost_dict.get(cat, 1.0))
    #print(f"Final boost for categories {cat_list} at hour {hour}: {boost:.2f}")
    # number between 1.0 and 1.5, where 1.0 means no boost and 1.5 means max boost
    return boost


# Ejemplo de uso
boost = context_score_by_hour(
    ["coffee_&_tea", "bakeries"],
    hour=8
)
print(f"Contextual boost for morning coffee: {boost:.2f}")

Contextual boost for morning coffee: 1.50


### Boost por categorias deseadas

In [12]:
def preferred_category_score(
    cat_list: list[str],
    preferred_categories: list[str],
) -> float:
    
    
    if preferred_categories is None:
        return 1.0
    
    cat_list_clean = {
        str(cat).lower().strip()
        for cat in cat_list
    }
    
    matches = cat_list_clean.intersection(preferred_categories)
    
    if len(matches) == 0:
        return 0.0
    
    return len(matches) / len(preferred_categories)

### Funcion de ponderacion

In [13]:
# Función de ponderación final para combinar los diferentes scores en un único score final
def restaurant_ponderate_score(
    model_score: float,
    popularity_score: float,
    hour_score: float = 0.0,
    city_score: float = 0.0,
    category_score: float = 0.0,
    use_hour: bool = True,
    use_city: bool = True,
    use_category: bool = True
) -> float:
    
    weights = {
        "model_score": 0.60,
        "popularity_score": 0.10,
        "hour_score": 0.10 if use_hour else 0.0,
        "city_score": 0.05 if use_city else 0.0,
        "category_score": 0.05 if use_category else 0.0,
    }

    total_weight = sum(weights.values())

    normalized_weights = {
        key: value / total_weight
        for key, value in weights.items()
    }

    score = (
        normalized_weights["model_score"] * model_score +
        normalized_weights["popularity_score"] * popularity_score +
        normalized_weights["hour_score"] * hour_score +
        normalized_weights["city_score"] * city_score +
        normalized_weights["category_score"] * category_score
    )

    return score * 5  # Escalamos de vuelta a [0, 5]


### Funcion principal de recomendacion

In [35]:
def normalize_category_list(
    categories: Optional[Union[str, list[str]]]
) -> Optional[list[str]]:

    if categories is None:
        return None

    if isinstance(categories, str):
        categories = [categories]

    return [
        category.lower().strip()
        for category in categories
    ]

def minmax_1d(values):
    values = np.asarray(values).reshape(-1, 1)
    return MinMaxScaler().fit_transform(values).ravel()

# MAIN FUNCTION
def recommend_contextual_businesses(
    user_id: str,
    model_svd,
    model_contextual,
    df_businesses: pd.DataFrame,
    df_reviews: pd.DataFrame,
    top_n: int = 10,
    preferred_cities: Optional[Union[str, list[str]]] = None,
    filter_by_city: bool = False,
    preferred_categories: Optional[Union[str, list[str]]] = None,
    filter_by_categories: bool = False,
    preferred_hour: Optional[int] = 20, # entero entre 0 y 23
    filter_by_open_hour: bool = False,
    preferred_day: Optional[str] = None,
    filter_by_open_day: bool = False,

) -> pd.DataFrame:

    preferred_categories = normalize_category_list(preferred_categories)
    preferred_cities = normalize_category_list(preferred_cities)# Crear 5 variables de intención

    if "category_list" not in df_businesses.columns:
        df_businesses["category_list"] = df_businesses["categories"].fillna("").apply(clean_categories)

    # Crear variables de intención para cada grupo de categorías
    for intent_name, intent_categories in INTENT_GROUPS.items():
        df_businesses[f"intent_{intent_name}"] = df_businesses["category_list"].apply(
            lambda categories: has_any_category(categories, intent_categories)
        )

    
    # 1. Negocios ya vistos por el usuario
    seen_businesses = set(
        df_reviews.loc[
            df_reviews["user_id"] == user_id,
            "business_id"
        ]
    )

    # 2. Filtro base: bussiness abiertos oficialmente y no vistos por el usuario
    base_mask = (
        ~df_businesses["business_id"].isin(seen_businesses) &
        (df_businesses["is_open"] == 1)
    )

    candidates = df_businesses[base_mask].copy()

    # 3. Filtro opcional por ciudad
    # Si activo el filtro, elimina de la lista de candidatos aquellos negocios cuya ciudad no esté en preferred_cities
    if preferred_cities is not None and filter_by_city:

        candidates = candidates[
            candidates["city"]
            .str.lower()
            .str.strip()
            .isin(preferred_cities)
        ].copy()

    # 4. Filtro opcional por categorías deseadas
    # Si activo el filtro, elimina de la lista de candidatos aquellos negocios que no tengan al menos una categoría en preferred_categories
    if preferred_categories is not None and filter_by_categories:
        candidates = candidates[
            candidates["category_list"].apply(
                lambda cat_list: len(
                    {
                        str(cat).lower().strip()
                        for cat in cat_list
                    }.intersection(preferred_categories)
                ) > 0
            )
        ].copy()

    # 5. Filtro opcional por día, hora o ambos
    # Si activo el filtro, elimina de la lista de candidatos aquellos negocios que no estén abiertos en el listado de dias y/o hora preferidos
    if "hours" in candidates.columns:
        candidates = candidates[
            candidates["hours"].apply(
                lambda hours_dict: is_open_at_context(
                    hours_dict=hours_dict,
                    day_list=preferred_day,
                    hour=preferred_hour,
                    filter_by_day=filter_by_open_day,
                    filter_by_hour=filter_by_open_hour
                )
            )
        ].copy()

    if candidates.empty:
        return pd.DataFrame()

    # 6. Score del modelo SVD++
    candidates["pred_rating"] = candidates["business_id"].apply(
        lambda business_id: model_svd.predict(user_id, business_id).est
    )

    # 7. Score por hora
    if preferred_hour is not None:
        candidates["hour_score"] = candidates["category_list"].apply(
            lambda cat_list: context_score_by_hour(cat_list, preferred_hour)
        )
    else:
        candidates["hour_score"] = 1.0

    # 8. Score por ciudad
    candidates["city_score"] = candidates["city"].apply(
        lambda city: city_match_score(
            city=city,
            preferred_cities=preferred_cities
        )
    )

    # 9. Score por categorías deseadas
    candidates["category_score"] = candidates["category_list"].apply(
        lambda cat_list: preferred_category_score(
            cat_list=cat_list,
            preferred_categories=preferred_categories
        )
    )
    # 10. Score de popularidad (puede ser simplemente el review_count normalizado)
    max_review_count = candidates["review_count"].max()
    candidates["popularity_score"] = candidates["review_count"] / max_review_count if max_review_count > 0 else 0

    # 11. otro contexto score con el modelo contextual entrenado (que puede incluir interacciones complejas entre las variables de contexto)
    
    intent_features = [col for col in candidates.columns if col.startswith("intent_")]
    num_features = ['pred_rating', "longitude", "latitude", "review_count"] 
    features = num_features + intent_features


    candidates["pred_contextual_rating"] =  model_contextual.predict(candidates[features])
    candidates["model_score"] = minmax_1d(candidates["pred_contextual_rating"])

    # 11. Score final
    candidates["final_score"] = candidates.apply(
        lambda row: restaurant_ponderate_score(
            model_score=row["model_score"],
            popularity_score=row["popularity_score"],
            hour_score=row["hour_score"],
            city_score=row["city_score"],
            category_score=row["category_score"],
            use_hour=preferred_hour is not None,
            use_city=preferred_cities is not None,
            use_category=preferred_categories is not None
        ),
        axis=1
    )

    
    recommendations = candidates.sort_values(
        "final_score",
        ascending=False
    ).head(top_n)

    columns_to_return = [
        "business_id",
        "name",
        "city",
        "category_list",
        "pred_rating",
        "final_score"
    ]

    if "hours" in recommendations.columns:
        columns_to_return.append("hours")

    return recommendations[columns_to_return]

In [15]:
model_svd = joblib.load(MODEL_DIR / 'model_SVDpp_100.joblib')
model_contextual =  joblib.load(MODEL_DIR / 'model_contextual.joblib')

no filtrar ni por dia ni por hora ni por ciudad

In [36]:
recommendations = recommend_contextual_businesses(
    user_id="Ha3iJu77CxlrFm-vQRs_8g",
    model_svd=model_svd,
    model_contextual=model_contextual,
    df_businesses=df_businesses,
    df_reviews=df_reviews,
    filter_by_open_day=False,
    filter_by_open_hour=False,
    filter_by_city=False,
    filter_by_categories=False,
)
recommendations

,business_id,name,city,category_list,pred_rating,final_score,hours
40360,fq1yCVBgBB7s6V-D68NO1g,Cafe Mi Quang,Philadelphia,"[restaurants, vietnamese]",4.899989,4.436851,"{'Monday': '8:30-19:0', 'Wednesday': '8:30-19:..."
74926,brmx8fQ7JQMHgfU_Req7HA,Gonzo's Smokehouse & BBQ,Luling,"[restaurants, food, barbeque, pop-up_restauran...",4.825893,4.411709,"{'Thursday': '11:30-15:0', 'Friday': '11:30-15..."
143157,ytynqOUb3hjKeJfRj5Tshw,Reading Terminal Market,Philadelphia,"[candy_stores, shopping, department_stores, fa...",4.567751,4.398077,"{'Monday': '8:0-18:0', 'Tuesday': '8:0-18:0', ..."
37284,0LBWe0PB3rimYdcRKJCRdQ,Taco De Oro,Lutz,"[tacos, food, mexican, restaurants, food_trucks]",4.867409,4.395226,"{'Monday': '10:0-21:0', 'Tuesday': '10:0-21:0'..."
125597,OR7VJQ3Nk1wCcIbPN4TCQQ,Smiling With Hope Pizza,Reno,"[italian, restaurants, salad, pizza]",4.790421,4.387298,"{'Monday': '0:0-0:0', 'Wednesday': '17:0-20:0'..."
63807,k_3KZQcs0H1wTAa0Odzhtw,Ecotech Import Auto Service,Saint Louis,"[automotive, auto_repair]",4.911419,4.377504,"{'Monday': '8:0-18:0', 'Tuesday': '8:0-18:0', ..."
44048,eqK66D5jQRO2eY1_MhAo_Q,Dan's Fresh Meats,Philadelphia,"[food, butcher]",4.964571,4.377478,"{'Monday': '9:0-17:0', 'Tuesday': '9:0-17:0', ..."
3968,j4fMC0VXYevNhs7wM9LSug,Armstrong Locksmith,Nashville,"[home_services, keys_&_locksmiths]",4.910243,4.376602,"{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:0', '..."
14070,STEG37SqBC3PkwY4wgSoPg,Taylor Home Solutions,Nashville,"[carpeting, air_duct_cleaning, heating_&_air_c...",4.909356,4.375978,"{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:0', '..."
27407,cVV8GWVIe9BwyCOKwrFgPA,Castellino's,Philadelphia,"[restaurants, food, sardinian, italian, delis,...",4.780274,4.374100,"{'Monday': '0:0-0:0', 'Tuesday': '12:0-17:0', ..."


filtrar por ciudad

In [38]:
recommendations = recommend_contextual_businesses(
    user_id="Ha3iJu77CxlrFm-vQRs_8g",
    model_svd=model_svd,
    model_contextual=model_contextual,
    df_businesses=df_businesses,
    df_reviews=df_reviews,
    preferred_cities=["Philadelphia", "Tampa", "Nashville"],
    filter_by_city=True,
    filter_by_open_day=False,
    filter_by_open_hour=False,
    top_n=10
)

recommendations[["name", "city", "category_list", "pred_rating", "final_score"]]

,name,city,category_list,pred_rating,final_score
143157,Reading Terminal Market,Philadelphia,"[candy_stores, shopping, department_stores, fa...",4.567751,4.570543
40360,Cafe Mi Quang,Philadelphia,"[restaurants, vietnamese]",4.899989,4.500687
91757,Hattie B’s Hot Chicken - Nashville,Nashville,"[american_(traditional), chicken_shop, souther...",4.439937,4.447968
44048,Dan's Fresh Meats,Philadelphia,"[food, butcher]",4.964571,4.444073
3968,Armstrong Locksmith,Nashville,"[home_services, keys_&_locksmiths]",4.910243,4.443512
14070,Taylor Home Solutions,Nashville,"[carpeting, air_duct_cleaning, heating_&_air_c...",4.909356,4.443037
27407,Castellino's,Philadelphia,"[restaurants, food, sardinian, italian, delis,...",4.780274,4.441684
122726,Keson Thai Restaurant,Tampa,"[thai, seafood, restaurants, fast_food]",4.783259,4.435290
73935,Rittenhouse Acupuncture,Philadelphia,"[acupuncture, traditional_chinese_medicine, he...",4.894077,4.431169
84866,Burritos La Mina,Nashville,"[food, restaurants, food_trucks, mexican]",4.830809,4.425711


Ciudad como boost, no como filtro
- Aquí pueden salir restaurantes de otras ciudades, pero las ciudades preferidas reciben mejor `city_score`.

In [39]:
recommendations = recommend_contextual_businesses(
    user_id="Ha3iJu77CxlrFm-vQRs_8g",
    model_svd=model_svd,
    model_contextual=model_contextual,
    df_businesses=df_businesses,
    df_reviews=df_reviews,
    preferred_cities=["tampa"],
    filter_by_city=False,  # no filtramos por ciudad, solo le damos un boost a las que están en esas ciudades
    filter_by_open_day=False,
    filter_by_open_hour=False,
    top_n=10
)

recommendations[["name", "city", "category_list", "pred_rating", "final_score"]]

,name,city,category_list,pred_rating,final_score
122726,Keson Thai Restaurant,Tampa,"[thai, seafood, restaurants, fast_food]",4.783259,4.434857
150173,Massage & Bodywork By Kuryn,Tampa,"[massage_therapy, beauty_&_spas, massage, heal...",4.918461,4.417710
142749,Terra Gaucha Brazilian Steakhouse - Tampa,Tampa,"[steakhouses, restaurants, buffets, brazilian,...",4.685326,4.414824
108547,The Ravioli Company,Tampa,"[restaurants, event_planning_&_services, cater...",4.733437,4.405041
118708,The Mediterranean Chickpea,Tampa,"[restaurants, vegetarian, mediterranean, vegan]",4.801890,4.402373
71573,Amaretto Ristorante,Tampa,"[italian, restaurants]",4.742987,4.395268
58300,Yolk White & Associates,Tampa,"[food_stands, restaurants, food, coffee_&_tea,...",4.783222,4.385971
149077,Pure Kitchen Organic Vegan,Tampa,"[vegetarian, juice_bars_&_smoothies, organic_s...",4.786561,4.384465
11490,Chicago Paulies,Tampa,"[hot_dogs, vegetarian, burgers, restaurants, f...",4.776485,4.377248
83015,Two Broke Spokes,Tampa,"[bikes, local_services, sporting_goods, bike_r...",4.877418,4.376614


filtrar solo por dia

In [40]:
preferred_day = "Friday"
recommendations = recommend_contextual_businesses(
    user_id="Ha3iJu77CxlrFm-vQRs_8g",
    model_svd=model_svd,
    model_contextual=model_contextual,
    df_businesses=df_businesses,
    df_reviews=df_reviews,
    preferred_day=[preferred_day],
    filter_by_open_day=True,
    filter_by_open_hour=False
)

recommendations[["name", "pred_rating", "final_score", "hours"]]

,name,pred_rating,final_score,hours
40360,Cafe Mi Quang,4.899989,4.436851,"{'Monday': '8:30-19:0', 'Wednesday': '8:30-19:..."
74926,Gonzo's Smokehouse & BBQ,4.825893,4.411709,"{'Thursday': '11:30-15:0', 'Friday': '11:30-15..."
143157,Reading Terminal Market,4.567751,4.398077,"{'Monday': '8:0-18:0', 'Tuesday': '8:0-18:0', ..."
37284,Taco De Oro,4.867409,4.395226,"{'Monday': '10:0-21:0', 'Tuesday': '10:0-21:0'..."
125597,Smiling With Hope Pizza,4.790421,4.387298,"{'Monday': '0:0-0:0', 'Wednesday': '17:0-20:0'..."
63807,Ecotech Import Auto Service,4.911419,4.377504,"{'Monday': '8:0-18:0', 'Tuesday': '8:0-18:0', ..."
44048,Dan's Fresh Meats,4.964571,4.377478,"{'Monday': '9:0-17:0', 'Tuesday': '9:0-17:0', ..."
3968,Armstrong Locksmith,4.910243,4.376602,"{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:0', '..."
14070,Taylor Home Solutions,4.909356,4.375978,"{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:0', '..."
27407,Castellino's,4.780274,4.374100,"{'Monday': '0:0-0:0', 'Tuesday': '12:0-17:0', ..."


filtrar solo por hora

In [41]:
recommendations = recommend_contextual_businesses(
    user_id="Ha3iJu77CxlrFm-vQRs_8g",
    model_svd=model_svd,
    model_contextual=model_contextual,
    df_businesses=df_businesses,
    df_reviews=df_reviews,
    preferred_hour=20, # int entre 0 y 23
    filter_by_open_day=False,
    filter_by_open_hour=True
)
recommendations[["name", "pred_rating", "final_score", "hours"]]

,name,pred_rating,final_score,hours
37284,Taco De Oro,4.867409,4.397200,"{'Monday': '10:0-21:0', 'Tuesday': '10:0-21:0'..."
125597,Smiling With Hope Pizza,4.790421,4.389212,"{'Monday': '0:0-0:0', 'Wednesday': '17:0-20:0'..."
3968,Armstrong Locksmith,4.910243,4.378634,"{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:0', '..."
14070,Taylor Home Solutions,4.909356,4.378008,"{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:0', '..."
27407,Castellino's,4.780274,4.376027,"{'Monday': '0:0-0:0', 'Tuesday': '12:0-17:0', ..."
122726,Keson Thai Restaurant,4.783259,4.370212,"{'Monday': '11:0-21:0', 'Tuesday': '11:0-21:0'..."
119161,Sterling Carpet Care,4.895367,4.368672,"{'Monday': '8:0-21:0', 'Tuesday': '8:0-21:0', ..."
98075,Sesami,4.829370,4.365259,"{'Monday': '0:0-0:0', 'Tuesday': '11:30-19:30'..."
105666,Steves iPhone Repair,4.891037,4.364301,"{'Monday': '0:0-0:0', 'Tuesday': '12:0-18:0', ..."
107311,Elevated Wellness,4.923934,4.363982,"{'Monday': '0:0-0:0', 'Wednesday': '10:0-17:30..."


filtrar por dia y hora

In [42]:
recommendations = recommend_contextual_businesses(
    user_id="Ha3iJu77CxlrFm-vQRs_8g",
    model_svd=model_svd,
    model_contextual=model_contextual,
    df_businesses=df_businesses,
    df_reviews=df_reviews,
    preferred_day=["Friday"],
    preferred_hour=20,
    filter_by_open_day=True,
    filter_by_open_hour=True
)
recommendations[["name", "pred_rating", "final_score", "hours"]]

,name,pred_rating,final_score,hours
37284,Taco De Oro,4.867409,4.425714,"{'Monday': '10:0-21:0', 'Tuesday': '10:0-21:0'..."
122726,Keson Thai Restaurant,4.783259,4.398041,"{'Monday': '11:0-21:0', 'Tuesday': '11:0-21:0'..."
119161,Sterling Carpet Care,4.895367,4.397793,"{'Monday': '8:0-21:0', 'Tuesday': '8:0-21:0', ..."
84866,Burritos La Mina,4.830809,4.388218,"{'Monday': '10:30-21:0', 'Tuesday': '10:30-21:..."
99968,Mio’s Grill & Cafe,4.825510,4.383875,"{'Monday': '0:0-0:0', 'Tuesday': '11:0-21:0', ..."
92813,Rooster Thai Sushi,4.767076,4.382890,"{'Monday': '0:0-0:0', 'Tuesday': '16:0-20:30',..."
87525,Pie12 Napoletana Coal Fired Pizzeria,4.773351,4.380026,"{'Monday': '16:0-20:0', 'Tuesday': '16:0-20:0'..."
142749,Terra Gaucha Brazilian Steakhouse - Tampa,4.685326,4.375790,"{'Monday': '17:0-21:30', 'Tuesday': '17:0-21:3..."
25746,El Volcan cocina mexicana,4.813163,4.373711,"{'Monday': '0:0-0:0', 'Tuesday': '10:30-21:0',..."
5212,Valente’s Cucina,4.728894,4.367463,"{'Monday': '0:0-0:0', 'Tuesday': '16:0-21:0', ..."


filtrar por categorias

In [43]:
recommendations = recommend_contextual_businesses(
    user_id="Ha3iJu77CxlrFm-vQRs_8g",
    model_svd=model_svd,
    model_contextual=model_contextual,
    df_businesses=df_businesses,
    df_reviews=df_reviews,
    preferred_day=["Friday"],
    preferred_hour=20,
    preferred_categories=["pizza", "italian", "mexican"],
    filter_by_open_day=True,
    filter_by_open_hour=True
)
recommendations[["name", "pred_rating", "final_score", "hours", "category_list"]]

,name,pred_rating,final_score,hours,category_list
87525,Pie12 Napoletana Coal Fired Pizzeria,4.773351,4.318456,"{'Monday': '16:0-20:0', 'Tuesday': '16:0-20:0'...","[pizza, restaurants, salad, italian]"
114768,Chiquita's Pizzeria & Mexican Grill,4.648912,4.297421,"{'Tuesday': '11:0-23:0', 'Wednesday': '11:0-23...","[mexican, pizza, restaurants, italian]"
37284,Taco De Oro,4.867409,4.263417,"{'Monday': '10:0-21:0', 'Tuesday': '10:0-21:0'...","[tacos, food, mexican, restaurants, food_trucks]"
28318,Bacio Kitchen + Catering,4.666919,4.234639,"{'Monday': '0:0-0:0', 'Tuesday': '16:0-21:0', ...","[burgers, specialty_food, caterers, american_(..."
117745,I Tre Mori,4.682153,4.232150,"{'Tuesday': '17:0-21:0', 'Wednesday': '17:0-21...","[bars, wine_bars, nightlife, restaurants, pizz..."
5763,Food + Drink,4.591522,4.230823,"{'Monday': '0:0-0:0', 'Wednesday': '16:0-22:0'...","[food, italian, cocktail_bars, nightlife, waff..."
84866,Burritos La Mina,4.830809,4.228127,"{'Monday': '10:30-21:0', 'Tuesday': '10:30-21:...","[food, restaurants, food_trucks, mexican]"
16519,Ravanesi Pizzeria Napoletana,4.665676,4.224278,"{'Tuesday': '16:30-21:0', 'Wednesday': '16:30-...","[pizza, italian, restaurants]"
25746,El Volcan cocina mexicana,4.813163,4.214473,"{'Monday': '0:0-0:0', 'Tuesday': '10:30-21:0',...","[mexican, restaurants]"
5212,Valente’s Cucina,4.728894,4.208593,"{'Monday': '0:0-0:0', 'Tuesday': '16:0-21:0', ...","[italian, farmers_market, tapas/small_plates, ..."
